<a href="https://github.com/N3iKos/SMFactory">
  <img alt="GitHub repo" src="https://img.shields.io/badge/GitHub-6e5494?style=for-the-badge&logo=github&logoColor=white"/>
</a><br>

*   get your civitai api key from [here](https://civitai.com/user/account)


In [ ]:
#@title <b><font color='orange'>WebUI Installer</font></b> {"display-mode":"form"}
Webui = 'Forge-Neo' #@param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
Civitai__Key = '' #@param { type: "string", placeholder: "Your Civitai API Key (required)" }
HF_Read_Token = '' #@param { type: "string", placeholder: "Your Huggingface READ Token (optional)" }
Mount_GDrive = 'No' #@param ["Yes", "No"]

mount = Mount_GDrive
if mount == 'Yes':
    from google.colab import drive
    drive.mount('/content/drive')

!curl -sLo /content/setup.py https://github.com/N3iKos/SMFactory/raw/main/script/KC/setup.py
%run /content/setup.py --webui="$Webui" --civitai_key="$Civitai__Key" --hf_read_token="$HF_Read_Token"

if mount == 'Yes':
    from pathlib import Path
    d = Path('/content/drive/MyDrive/SMFactory')
    d.mkdir(parents=True, exist_ok=True)
    for n, p in {'checkpoint': CKPT, 'lora': LORA, 'vae': VAE, 'embeddings': Embeddings}.items():
        f = d / n
        f.mkdir(parents=True, exist_ok=True)
        s = p / f'drive-{n}'
        if not s.exists():
            s.symlink_to(f, target_is_directory=True)

    if 'WebUI_Output' in globals() and not WebUI_Output.exists():
        o = d / {'ComfyUI': 'comfyui-output', 'SwarmUI': 'swarmui-output'}.get(Webui, 'output')
        o.mkdir(parents=True, exist_ok=True)
        WebUI_Output.symlink_to(o, target_is_directory=True)

    if Webui not in {'ComfyUI', 'SwarmUI'}:
        wc = WebUI / 'cache'
        c = d / 'cache'
        c.mkdir(parents=True, exist_ok=True)
        if not wc.exists():
            wc.symlink_to(c, target_is_directory=True)


In [ ]:
#@title <b><font color='orange'>Model Downloader</font></b> {"display-mode":"form"}
Checkpoint_1 = "" #@param {type:"string"}
Checkpoint_2 = "" #@param {type:"string"}
Checkpoint_3 = "" #@param {type:"string"}
Checkpoint_4 = "" #@param {type:"string"}
Checkpoint_5 = "" #@param {type:"string"}
Lora_1 = "" #@param {type:"string"}
Lora_2 = "" #@param {type:"string"}
Lora_3 = "" #@param {type:"string"}
Lora_4 = "" #@param {type:"string"}
Lora_5 = "" #@param {type:"string"}
VAE_URL = "" #@param {type:"string"}
Load_from_Drive = False #@param {type:"boolean"}
Parallel_Download = False #@param {type:"boolean"}
Max_Workers = 3 #@param {type:"slider", min:1, max:10, step:1}

downloads = []
if Checkpoint_1.strip(): downloads.append((Checkpoint_1, CKPT))
if Checkpoint_2.strip(): downloads.append((Checkpoint_2, CKPT))
if Checkpoint_3.strip(): downloads.append((Checkpoint_3, CKPT))
if Checkpoint_4.strip(): downloads.append((Checkpoint_4, CKPT))
if Checkpoint_5.strip(): downloads.append((Checkpoint_5, CKPT))

if Lora_1.strip(): downloads.append((Lora_1, LORA))
if Lora_2.strip(): downloads.append((Lora_2, LORA))
if Lora_3.strip(): downloads.append((Lora_3, LORA))
if Lora_4.strip(): downloads.append((Lora_4, LORA))
if Lora_5.strip(): downloads.append((Lora_5, LORA))

if VAE_URL.strip(): downloads.append((VAE_URL, VAE))

if downloads:
    from nenen88 import download_list
    download_list(downloads, load_from_drive=Load_from_Drive, parallel=Parallel_Download, max_workers=Max_Workers)


In [ ]:
#@title <b><font color='orange'>Extra Assets Downloader</font></b> {"display-mode":"form"}
Extension_1 = "" #@param {type:"string"}
Extension_2 = "" #@param {type:"string"}
Extension_3 = "" #@param {type:"string"}
Extension_4 = "" #@param {type:"string"}
Extension_5 = "" #@param {type:"string"}
Embedding_1 = "" #@param {type:"string"}
Embedding_2 = "" #@param {type:"string"}
Embedding_3 = "" #@param {type:"string"}
Upscaler_1 = "" #@param {type:"string"}
Upscaler_2 = "" #@param {type:"string"}
Upscaler_3 = "" #@param {type:"string"}
Load_from_Drive = False #@param {type:"boolean"}
Assets_Parallel_Download = False #@param {type:"boolean"}
Assets_Max_Workers = 3 #@param {type:"slider", min:1, max:10, step:1}

import os
import shutil
from pathlib import Path

gdrive_mounted = Path('/content/drive/MyDrive').exists()
drive_ext_dir = Path('/content/drive/MyDrive/SMFactory/extensions') if (Load_from_Drive and gdrive_mounted) else None

extensions = [Extension_1, Extension_2, Extension_3, Extension_4, Extension_5]
extensions = [e.strip() for e in extensions if e.strip()]

if extensions:
    if Load_from_Drive and gdrive_mounted:
        drive_ext_dir.mkdir(parents=True, exist_ok=True)
        to_clone = []
        for ext_url in extensions:
            repo_name = ext_url.split('/')[-1].replace('.git', '')
            drive_repo = drive_ext_dir / repo_name
            local_repo = Extensions / repo_name
            if drive_repo.exists():
                print(f"[SMFactory] Extension found in Google Drive: {repo_name}. Symlinking...")
                if local_repo.exists() or local_repo.is_symlink():
                    try:
                        if local_repo.is_symlink() or local_repo.is_file(): local_repo.unlink()
                        elif local_repo.is_dir(): shutil.rmtree(local_repo)
                    except Exception: pass
                local_repo.symlink_to(drive_repo)
            else:
                to_clone.append((ext_url, drive_repo, local_repo))
        if to_clone:
            for ext_url, drive_repo, local_repo in to_clone:
                os.chdir(drive_ext_dir)
                %clone $ext_url
                if drive_repo.exists():
                    if local_repo.exists() or local_repo.is_symlink():
                        try:
                            if local_repo.is_symlink() or local_repo.is_file(): local_repo.unlink()
                            elif local_repo.is_dir(): shutil.rmtree(local_repo)
                        except Exception: pass
                    local_repo.symlink_to(drive_repo)
            os.chdir(HOME)
    else:
        os.chdir(Extensions)
        for ext_url in extensions:
            %clone $ext_url
        os.chdir(HOME)

asset_downloads = []
if Embedding_1.strip(): asset_downloads.append((Embedding_1, Embeddings))
if Embedding_2.strip(): asset_downloads.append((Embedding_2, Embeddings))
if Embedding_3.strip(): asset_downloads.append((Embedding_3, Embeddings))
if Upscaler_1.strip(): asset_downloads.append((Upscaler_1, Upscalers))
if Upscaler_2.strip(): asset_downloads.append((Upscaler_2, Upscalers))
if Upscaler_3.strip(): asset_downloads.append((Upscaler_3, Upscalers))

if asset_downloads:
    from nenen88 import download_list
    download_list(asset_downloads, load_from_drive=Load_from_Drive, parallel=Assets_Parallel_Download, max_workers=Assets_Max_Workers)


In [ ]:
#@title <b><font color='orange'>FLUX Model Downloader</font></b> {"display-mode":"form"}
FLUX_Variant = 'FLUX.1-schnell' #@param ["FLUX.1-schnell", "FLUX.1-dev"]
FLUX_Unet = "" #@param {type:"string"}
FLUX_Clip_L = "" #@param {type:"string"}
FLUX_T5XXL = "" #@param {type:"string"}
FLUX_VAE = "" #@param {type:"string"}
Load_from_Drive = False #@param {type:"boolean"}
Parallel_FLUX_Download = False #@param {type:"boolean"}
FLUX_Max_Workers = 2 #@param {type:"slider", min:1, max:10, step:1}

downloads = []
unet_url = FLUX_Unet.strip() if FLUX_Unet.strip() else (
    'https://huggingface.co/lllyasviel/flux1-dev-channel-last/resolve/main/flux1-schnell-fp8.safetensors'
    if FLUX_Variant == 'FLUX.1-schnell'
    else 'https://huggingface.co/lllyasviel/flux1-dev-channel-last/resolve/main/flux1-dev-fp8.safetensors'
)
downloads.append((unet_url, UNET))

clip_l_url = FLUX_Clip_L.strip() if FLUX_Clip_L.strip() else 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors'
downloads.append((clip_l_url, CLIP))

t5xxl_url = FLUX_T5XXL.strip() if FLUX_T5XXL.strip() else 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors'
downloads.append((t5xxl_url, CLIP))

vae_url = FLUX_VAE.strip() if FLUX_VAE.strip() else 'https://huggingface.co/lllyasviel/flux1-dev-channel-last/resolve/main/ae.safetensors'
downloads.append((vae_url, VAE))

if downloads:
    from nenen88 import download_list
    download_list(downloads, load_from_drive=Load_from_Drive, parallel=Parallel_FLUX_Download, max_workers=FLUX_Max_Workers)


In [ ]:
#@title <b><font color='orange'>ControlNet Downloader Widget</font></b> {"display-mode":"form"}
''' Controlnet '''
%run $Controlnet_Widget


In [ ]:
#@title <b><font color='orange'>Launcher WebUI</font></b> {"display-mode":"form"}
Software = 'Forge-Neo' #@param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
Ngrok_Token = "" #@param {type:"string"}
Zrok_Token = "" #@param {type:"string"}
Extra_Args = "" #@param {type:"string"}
Skip_ComfyUI_Check = False #@param {type:"boolean"}
Skip_Widget = False #@param {type:"boolean"}

%cd -q $WebUI

args_str = f" {Extra_Args.strip()}"
if Skip_ComfyUI_Check:
    args_str += " --skip-comfyui-check"
if Skip_Widget:
    args_str += " --skip-widget"
if Ngrok_Token.strip():
    args_str += f" --N={Ngrok_Token.strip()}"
if Zrok_Token.strip():
    args_str += f" --Z={Zrok_Token.strip()}"

import json
from pathlib import Path

marking_path = Path(HOMEPATH) / 'gutris1/marking.json' if 'HOMEPATH' in globals() else Path.home() / '.gutris1/marking.json'
if marking_path.exists():
    d = json.loads(marking_path.read_text())
    d['ui'] = Software
    marking_path.write_text(json.dumps(d, indent=4))

%run segsmaker.py $args_str
